<div align="center">

# 🚕 Yandex Go Analytics

## 2 · Batch Ingestion

*Read Parquet and Avro files straight from the Volume and persist them in Bronze — as they are.*

![layer](https://img.shields.io/badge/layer-bronze-CD7F32)
![ingestion](https://img.shields.io/badge/ingestion-Auto%20Loader-FF3621)
![formats](https://img.shields.io/badge/formats-Parquet%20%2B%20Avro-00ADD4)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [1 · Setup](./1.%20Setup.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | [3 · Streaming Ingestion & Delta Features](./3.%20Streaming%20Ingestion%20%26%20Delta%20Features.ipynb) |

**Progress** &nbsp; ▰▰▱▱▱▱▱▱  **2 / 8**

---

## 🎯 Task requirements

- [x] Read the **`.parquet`** and **`.avro`** files directly from the Volume
- [x] Write them **"as-is"** into the `bronze` schema as **Delta** tables: `bronze.taxi` and `bronze.users`

## 🧩 How it is solved

Both tables use the same service, `BaseIngestionService`:

| Step | What happens |
|:--|:--|
| 1 · **Extract** | Auto Loader (`cloudFiles`) reads new files from the Volume; the schema is inferred and stored next to the checkpoint |
| 2 · **Audit columns** | `_ingested_at` (timestamp) and `_source_file` (path) are added — the business columns stay untouched |
| 3 · **Load** | Append to a Delta table with `trigger(availableNow=True)`: process everything available, then stop |

| Entity | Source | Target |
|:--|:--|:--|
| 🚕 Trips | `…/landing/parquet` | `bronze.taxi` |
| 👤 Users | `…/landing/avro` | `bronze.users` |

> 💡 **Incremental by design** — the checkpoint remembers which files were processed, so a re-run ingests only
> *new* files and never duplicates data. Auto Loader also adds a `_rescued_data` column that captures values
> that do not match the inferred schema.

## 1️⃣ Connect to Spark

In [1]:
from src.core.config import get_catalog, get_environment
from src.core.infra import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-go-batch-ingestion")
conn.connect()
spark = conn.spark

CATALOG = get_catalog()
print(f"✅ Connected | Spark {spark.version} | catalog = {CATALOG} | env = {get_environment()}")

2026-10-09 12:27:03.065 | INFO     | src.core.infra.connection.databricks_connector:connect:56 - SparkSession ready | app=yandex-go-batch-ingestion | version=4.2.0 | runtime=serverless-connect


✅ Connected | Spark 4.2.0 | catalog = yandex_go_dev | env = env


## 2️⃣ Peek at the raw sources

Spark can read the files directly from the Volume path — no copy, no staging.

### 🚕 Parquet — trips

In [5]:
SRC_PARQUET = f"/Volumes/{CATALOG}/raw_files/landing/parquet"
SRC_AVRO = f"/Volumes/{CATALOG}/raw_files/landing/avro"

source_trips = spark.read.parquet(SRC_PARQUET)
print(f"Parquet source: {source_trips.count():,} rows | {len(source_trips.columns)} columns")
display(source_trips.toPandas())

Parquet source: 20,000 rows | 22 columns


,id,user_id,driver_id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,...,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,e663308a-db8f-4a8c-8caf-7b213a6924b5,def57689-68f4-4bce-a4e7-5e8eb8f21423,760b3c5e-f4c3-4307-a72f-666d35f9377f,2,2026-10-06 10:22:04,2026-10-06 10:37:04,2,21.09,4,Y,...,1,58.47,2.5,0.5,8.77,0.0,1.0,73.74,2.5,0.00
1,cfc8a322-cd4d-4ed5-b184-da91f3b7c4d8,82dc4c8e-36b5-429a-acf5-e81e71316269,ff9a1b80-5ced-4e2e-b17f-6920daaafe5c,1,2026-10-06 11:31:48,2026-10-06 11:40:48,3,15.06,4,Y,...,1,59.67,0.0,0.5,8.95,0.0,1.0,71.37,0.0,1.25
2,f302fa37-91bb-4901-99d0-891626844d45,f8cda88b-436d-46e2-b83c-fe0be037e5ed,760b3c5e-f4c3-4307-a72f-666d35f9377f,1,2026-10-06 03:02:08,2026-10-06 03:18:08,3,26.83,1,N,...,3,71.76,0.5,0.5,10.76,0.0,1.0,84.52,0.0,0.00
3,3f6bb12c-08d1-4f70-96b4-ed74605c9322,0986bbeb-f23e-423d-8b9b-d93423c86d30,bbb4f560-f222-470c-b003-25331119d533,1,2026-10-05 19:23:45,2026-10-05 20:03:45,3,1.44,1,Y,...,4,6.83,0.0,0.5,1.02,0.0,1.0,11.85,2.5,0.00
4,30ee8831-0ea3-480b-af2a-a596820c8e9c,680bac63-b856-4035-bdc9-829015eabb27,4068cacc-1f7d-446f-9716-8ab3a4351379,2,2026-10-07 00:20:23,2026-10-07 00:33:23,2,24.44,1,N,...,1,96.19,0.5,0.5,14.43,0.0,1.0,115.12,2.5,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,2883a50a-5696-47a4-acaf-ebc5db33c6f6,118a9d29-2f92-4996-99f1-95d014822f53,1c7d03db-dcfa-4897-a030-1c524d00fd4a,2,2026-10-01 10:04:58,2026-10-01 10:17:58,3,0.61,4,N,...,1,5.18,1.0,0.5,0.78,0.0,1.0,12.21,2.5,1.25
19996,e92cb700-e4cf-46cb-8ca3-3cf3127e2a27,14fcdd54-9e8f-4965-8a2c-827e98326856,12bb6a79-4552-41f7-ac98-13408cb287fc,1,2026-10-01 12:06:40,2026-10-01 12:48:40,5,4.75,3,N,...,1,20.04,2.5,0.5,3.01,0.0,1.0,30.80,2.5,1.25
19997,01ac4b10-3896-4027-a19f-5183b4454e9c,def57689-68f4-4bce-a4e7-5e8eb8f21423,ba7858bb-61d4-41c5-a16c-5c3c2cab43d8,1,2026-10-01 12:44:20,2026-10-01 13:29:20,1,12.75,3,N,...,1,43.15,0.5,0.5,6.47,0.0,1.0,51.62,0.0,0.00
19998,e370561d-7628-4295-9534-e6dbc11bed3f,a14923c2-f920-464c-8763-fcd01f15c7b6,299c3bcf-90ef-4625-99fb-cdb5c6042444,1,2026-10-01 11:50:26,2026-10-01 12:26:26,3,20.85,1,Y,...,2,67.32,0.5,0.5,0.00,0.0,1.0,71.82,2.5,0.00


### 👤 Avro — users

In [6]:
source_users = spark.read.format("avro").load(SRC_AVRO)
print(f"Avro source: {source_users.count():,} rows | {len(source_users.columns)} columns")
display(source_users.toPandas())

Avro source: 4,000 rows | 5 columns


,id,full_name,email,registration_date,is_plus_subscriber
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,Heather Landry,trodriguez@example.net,2023-11-25,False
1,23b8c1e9-3924-46de-beb1-3b9046685257,Sandra Brewer,unewman@example.org,2023-12-08,True
2,bd9c66b3-ad3c-4d6d-9a3d-1fa7bc8960a9,Sara Gonzales,brendabrown@example.org,2025-05-26,False
3,972a8469-1641-4f82-8b9d-2434e465e150,Raymond Reynolds,brownshelley@example.com,2022-08-29,False
4,17fc695a-07a0-4a6e-8822-e8f36c031199,Anthony Williamson,bobby54@example.net,2025-07-25,False
...,...,...,...,...,...
3995,d56ce8ea-1959-4b5a-a7a8-f6366a35df59,Thomas Williams,nmendez@example.com,2025-11-03,True
3996,f065c817-687a-4850-9d9b-62317d45d8ef,Andrea Lyons,molinacindy@example.net,2025-01-12,True
3997,b0b63bcf-0860-4833-879d-0cdaf396ea37,Claire Cortez,wbaker@example.net,2022-01-25,False
3998,71d79665-7181-4dcf-b79e-fc6e5edb0d3c,Katherine Coleman,mortonjessica@example.com,2020-10-15,False


## 3️⃣ Ingest into Bronze

Two services, one pattern. Each call blocks until the available files are fully processed.

```text
 landing/avro      ──▶ UserBatchIngestionService      ──▶ bronze.users
 landing/parquet   ──▶ TaxiTripBatchIngestionService  ──▶ bronze.taxi
```

In [4]:
from src.core.etl.bronze import (
    TaxiTripBatchIngestionService,
    UserBatchIngestionService,
)

user_service = UserBatchIngestionService()
user_service.run(spark)

taxi_service = TaxiTripBatchIngestionService()
taxi_service.run(spark)

print("✅ Batch ingestion finished")

2026-10-09 12:27:10.727 | INFO     | src.core.etl.bronze.base:run:123 - Start data ingestion for yandex_go_dev.bronze.users
2026-10-09 12:27:10.728 | INFO     | src.core.etl.bronze.base:extract:63 - Reading data [avro] from: /Volumes/yandex_go_dev/raw_files/landing/avro
2026-10-09 12:27:10.924 | INFO     | src.core.etl.bronze.base:load:99 - Writing data in table Delta: yandex_go_dev.bronze.users
2026-10-09 12:27:29.648 | INFO     | src.core.etl.bronze.base:load:111 - Successfully loaded in yandex_go_dev.bronze.users
2026-10-09 12:27:29.651 | INFO     | src.core.etl.bronze.base:run:123 - Start data ingestion for yandex_go_dev.bronze.taxi
2026-10-09 12:27:29.652 | INFO     | src.core.etl.bronze.base:extract:63 - Reading data [parquet] from: /Volumes/yandex_go_dev/raw_files/landing/parquet
2026-10-09 12:27:29.654 | INFO     | src.core.etl.bronze.base:load:99 - Writing data in table Delta: yandex_go_dev.bronze.taxi
2026-10-09 12:27:45.978 | INFO     | src.core.etl.bronze.base:load:111 - Su

✅ Batch ingestion finished


---

## ✅ Checkpoint

- [x] Parquet and Avro files read directly from the Volume
- [x] `bronze.taxi` and `bronze.users` written as Delta tables
- [x] Row counts match the sources; re-running adds nothing

> **Next up ▶ [3 · Streaming Ingestion & Delta Features](./3.%20Streaming%20Ingestion%20%26%20Delta%20Features.ipynb)** — stream drivers into Bronze, evolve the schema and travel through time.

<div align="center"><sub>Yandex Go Analytics · notebook 2 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>